# Local Inference with Ollama

- **Non-local (API) calls.** OpenAI, Anthropic, Google. Nothing to install beyond Python packages.
- **Local calls with Ollama.** Installing Ollama, running your first model, then calling it from Python.

By the end, you'll have classified the same dataset two ways (API + Ollama) and be able to compare them side by side.

## Learning objectives

1. Set up API credentials safely.
2. Write a unified function that classifies text using OpenAI, Anthropic, or Gemini.
3. Force models to return **structured** output instead of free text.
4. Handle **retries / rate limits** so a flaky network doesn't kill a long run.
5. Track **token usage and cost** before scaling up an API-based run.
6. Install Ollama and run your **first local model**, with zero prior setup.
7. Call a local Ollama model from Python using the **same pipeline shape** as the API calls.
8. Compare API vs. local models on **cost, privacy, and performance**.

## How this notebook is structured

Every model-calling function (API or Ollama) has a **MOCK mode** and a **LIVE mode**, controlled by one flag near the top. In MOCK mode nothing leaves your computer and no software needs to be installed a fake "model" simulates realistic responses so you can build and test the *entire* pipeline, including Ollama, before installing anything. Flip one flag to `False`, install the real software, and the same pipeline code runs for real.

**Run the whole notebook in MOCK mode first.** Then work through the installation steps for real, flip the flag, and re-run.


# Non-Local (API) Calls

## The privacy tradeoff

Every API provider is a **third party**. When you call their API, the text you're classifying (e.g., a classroom transcript, an open-ended survey response, a student essay) leaves your machine and is processed on their servers.

Before running real data through any API, ask:

- Does your IRB protocol / data use agreement allow sending this data to a third party?
- Does the provider **train on your data** by default? (Usually off by default for paid API usage, but this can vary.)
- Could you de-identify the text first and still answer your research question?
- Is there a **zero data retention (ZDR)** or enterprise agreement available if your data is sensitive?

In [ ]:

import os
import time
import json
import random
import hashlib
import subprocess
import shutil

from google.colab import ai
from datasets import load_dataset
import pandas as pd
from pydantic import BaseModel, Field
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type

random.seed(42)

# THE ONE FLAG THAT MATTERS
USE_MOCK = True   # <- set to False only once you have real API keys AND/OR Ollama installed

print(f"Running in {'MOCK' if USE_MOCK else 'LIVE'} mode.")


## API keys

**Never** hardcode API keys in a notebook, and never commit them to git (including private repos -- assume they leak eventually).

1. Create a file named `.env` in this folder.
2. Add your keys:
   ```
   OPENAI_API_KEY=sk-...
   ANTHROPIC_API_KEY=sk-ant-...
   GOOGLE_API_KEY=AI...
   ```
3. Add `.env` to `.gitignore` **before** you ever `git add .`
4. Load it with `python-dotenv` (below).

If you're on shared/HPC systems: don't put real keys in a job script others can read. Use a restricted-permission file (`chmod 600 .env`).


In [ ]:
if USE_MOCK:
    print("MOCK mode: no API keys needed, no network calls will be made.")
else:
    from dotenv import load_dotenv
    load_dotenv()
    api_keys = {
        "openai": os.environ.get("OPENAI_API_KEY"),
        "anthropic": os.environ.get("ANTHROPIC_API_KEY"),
        "google": os.environ.get("GOOGLE_API_KEY"),
        }
    missing = [k for k, v in api_keys.items() if not v]
    if missing:
        raise EnvironmentError(f"Missing API keys for: {missing}. Check your .env file.")


## The Data
Goodreads reviews and ratings

In [ ]:
reviews = [
    ("Absolutely loved this book. Could not put it down, finished it in two days.", 5),
    ("This was a decent read but the pacing dragged in the middle third.", 3),
    ("I could not get through this. The characters felt flat and the plot made no sense.", 1),
    ("A solid mystery novel with a satisfying twist at the end.", 4),
    ("Terrible. Would not recommend to anyone. Waste of time.", 0),
    ("Pretty good overall, some parts felt rushed but the ending redeemed it.", 3),
    ("One of the best books I've read this year. Beautifully written.", 5),
    ("It was fine. Nothing special, nothing terrible. Forgettable.", 2),
    ("The world-building was incredible but the dialogue felt stilted.", 3),
    ("I really disliked the main character's decisions throughout. Frustrating read.", 1),
    ("A masterpiece. Every chapter built on the last perfectly.", 5),
    ("Okay premise, poor execution. The ending felt rushed and unearned.", 2),
    ("Charming and funny, exactly what I needed. Would read again.", 4),
    ("Confusing structure made this hard to follow. Gave up halfway through.", 1),
    ("Solid entry in the series, though not as strong as the first book.", 3),
]

df = pd.DataFrame(reviews, columns=["review_text", "true_rating"])
df.head()

## The Prompt


In [ ]:
SYSTEM_PROMPT = (
    "You are a careful annotator rating book reviews. "
    "Given a review, predict the rating on a 0 (bad) to 5 (good) scale, "
    "matching how the reviewer likely rated the book themselves. "
    "Respond only in the requested structured format."
)

def build_user_prompt(review_text: str) -> str:
    return f"Book review:\n\"\"\"\n{review_text}\n\"\"\"\n\nPredict the rating."


## The Output

## Cost Analysis


## Performance Evaluation